# WildPulse image review

Upload JPEG or PNG images, run wildlife detection, then download the results. Runs on CPU. Model startup and larger batches can take several minutes. Images stay in your Colab session; this notebook does not publish them to GitHub. Colab storage is temporary.

Predictions cover animals, people and vehicles, not species. The default confidence of 0.5 is for review and is not a production recommendation.

## Set up
Run this cell once per Colab session.

In [ ]:
%pip -q install sparrow-engine==0.1.32 opencv-python-headless
from pathlib import Path
from urllib.request import urlopen
import importlib.util
script = Path("/content/run_images.py")
script.write_bytes(urlopen("https://raw.githubusercontent.com/Shreyakb301/WildPulse/main/run_images.py", timeout=60).read())
spec = importlib.util.spec_from_file_location("wildpulse_images", script)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)
print("WildPulse is ready")


## Upload images
Each upload starts a separate batch. Choose JPEG or PNG files from your computer.

In [ ]:
from google.colab import files
import tempfile
batch = Path(tempfile.mkdtemp(prefix="wildpulse_", dir="/content"))
images = batch/"images"
images.mkdir()
uploaded = files.upload()
for name, content in uploaded.items():
    target = images/Path(name).name
    if target.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
        raise ValueError("Please choose JPEG or PNG images")
    target.write_bytes(content)
print(f"Uploaded {len(uploaded)} images")


## Detect and preview
Run once for each uploaded batch. To run again, upload a new batch first. Unreadable images are listed in the report. Unexpected detection errors stop the run so an incomplete export cannot look complete.

In [ ]:
from IPython.display import display, Image
confidence = 0.5 # @param {type:"slider", min:0.1, max:1.0, step:0.1}
summary, bundle = workflow.run(images, batch/"results", threshold=confidence)
print(f"Processed {summary['successful']} images; {summary['failed']} unreadable")
for record in summary["records"]:
    if record["status"] == "ok":
        print(record["image"])
        display(Image(filename=str(bundle.parent/record["preview"]), width=800))


## Download results
The ZIP contains annotated previews, JSON results and CSV detections. Download before the session ends. Original uploads are not included in the ZIP.

In [ ]:
files.download(str(bundle))
